### Перевощиков Никита, ИУ5Ц-21М, ММО АСОИУ, Рубежный контроль №1

#### Номер задачи №1 (Задача №3)
#### Для набора данных проведите кодирование одного (произвольного) категориального признака с использованием метода "weight of evidence (WoE) encoding".

In [3]:
# Импортируем необходимые библиотеки
import pandas as pd
import numpy as np

In [4]:
# Загрузка данных
df = pd.read_csv('urban_street_food_vendor_survival_dataset.csv')

In [5]:
# Общая информация о датасете
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22000 entries, 0 to 21999
Data columns (total 20 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   vendor_id                        22000 non-null  object 
 1   city                             22000 non-null  object 
 2   zone_type                        22000 non-null  object 
 3   vendor_age_years                 21242 non-null  float64
 4   years_in_business                20963 non-null  float64
 5   food_category                    22000 non-null  object 
 6   license_status                   21350 non-null  object 
 7   avg_daily_revenue_inr            20062 non-null  float64
 8   avg_daily_customers              20747 non-null  float64
 9   monthly_stall_rent_inr           20890 non-null  float64
 10  num_helpers                      22000 non-null  int64  
 11  hours_open_per_day               21558 non-null  float64
 12  competition_within

In [6]:
# Первые 5 строк датасета
df.head()

,vendor_id,city,zone_type,vendor_age_years,years_in_business,food_category,license_status,avg_daily_revenue_inr,avg_daily_customers,monthly_stall_rent_inr,num_helpers,hours_open_per_day,competition_within_100m,monthly_health_inspection_score,had_fine_last_year,avg_monthly_rainfall_mm,season_of_observation,has_online_presence,customer_complaint_rate,vendor_survived
0,VSF-00001,Delhi,Industrial,34.0,10.98,Chinese,Licensed,332.20,10.0,0.00,1,10.00,4.0,65.00,0.0,103.08,Winter,0.0,0.12,1
1,VSF-00002,Kochi,University Area,33.0,5.53,Chaat,Licensed,3138.90,180.0,2112.10,1,6.42,3.0,NaN,1.0,350.00,Monsoon,1.0,0.22,1
2,VSF-00003,Hyderabad,Industrial,48.0,2.70,Desserts & Sweets,Licensed,1626.36,67.0,5487.75,3,11.80,0.0,NaN,0.0,66.64,Post-Monsoon,0.0,0.09,0
3,VSF-00004,Bengaluru,Transit Hub,NaN,9.77,Beverages,Licensed,3592.98,171.0,1960.79,3,12.53,NaN,36.38,1.0,107.30,Post-Monsoon,1.0,0.10,1
4,VSF-00005,Delhi,Transit Hub,51.0,21.24,South Indian,Licensed,2418.07,105.0,2720.82,4,6.07,5.0,71.05,0.0,166.57,Monsoon,0.0,0.08,0


In [7]:
# Проверка на наличие пропусков
print("Проверка на наличие пропусков:")
df.isnull().sum()

Проверка на наличие пропусков:


vendor_id                             0
city                                  0
zone_type                             0
vendor_age_years                    758
years_in_business                  1037
food_category                         0
license_status                      650
avg_daily_revenue_inr              1938
avg_daily_customers                1253
monthly_stall_rent_inr             1110
num_helpers                           0
hours_open_per_day                  442
competition_within_100m             884
monthly_health_inspection_score    7759
had_fine_last_year                  221
avg_monthly_rainfall_mm               0
season_of_observation                 0
has_online_presence                 193
customer_complaint_rate            2420
vendor_survived                       0
dtype: int64

In [8]:
# 1. Для категориальных столбцов (object) - заполняем модой или 'Unknown'
cat_cols = df.select_dtypes(include=['object']).columns
for col in cat_cols:
    if df[col].isnull().sum() > 0:
        mode_val = df[col].mode()
        if len(mode_val) > 0:
            df[col] = df[col].fillna(mode_val.iloc[0])
        else:
            df[col] = df[col].fillna('Unknown')

# 2. Для числовых столбцов - заполняем медианой (или 0 для счётных/бинарных)
num_cols = df.select_dtypes(include=['float64', 'int64']).columns

# Отдельно выделим бинарные/счётные признаки, где 0 логичен:
binary_or_count_cols = [
    'num_helpers',
    'had_fine_last_year',
    'has_online_presence',
    'vendor_survived'
]

for col in num_cols:
    if df[col].isnull().sum() > 0:
        if col in binary_or_count_cols:
            df[col] = df[col].fillna(0)
        else:
            median_val = df[col].median()
            df[col] = df[col].fillna(median_val)

# Проверка: должны остаться 0 пропусков
print("Пропуски после очистки:")
df.isnull().sum()

Пропуски после очистки:


vendor_id                          0
city                               0
zone_type                          0
vendor_age_years                   0
years_in_business                  0
food_category                      0
license_status                     0
avg_daily_revenue_inr              0
avg_daily_customers                0
monthly_stall_rent_inr             0
num_helpers                        0
hours_open_per_day                 0
competition_within_100m            0
monthly_health_inspection_score    0
had_fine_last_year                 0
avg_monthly_rainfall_mm            0
season_of_observation              0
has_online_presence                0
customer_complaint_rate            0
vendor_survived                    0
dtype: int64

In [9]:
# Бинарная целевая переменная
threshold = df['avg_daily_revenue_inr'].median()
df['high_revenue'] = (df['avg_daily_revenue_inr'] > threshold).astype(int)

print(f"Медиана средней дневной выручки: {threshold:.2f} INR")
print(df['high_revenue'].value_counts(normalize=True))

Медиана средней дневной выручки: 1516.59 INR
high_revenue
0    0.544045
1    0.455955
Name: proportion, dtype: float64


In [10]:
# Подсчёт частот городов
city_counts = df['city'].value_counts()
print("Частоты городов (топ 10):")
print(city_counts.head(10))

# Определяем порог для "Other": например, меньше 30 записей, объединяем
min_count_for_keep = 30
common_cities = city_counts[city_counts >= min_count_for_keep].index.tolist()
df['city_group'] = df['city'].where(df['city'].isin(common_cities), 'Other')

# Вычисляем WoE с добавлением smoothing (eps = 0.5)
total_events = df['high_revenue'].sum()
total_non_events = len(df) - total_events
eps = 0.5

woe_data = (
    df.groupby('city_group')['high_revenue']
    .agg(count='count', events='sum')
    .reset_index()
)
woe_data['non_events'] = woe_data['count'] - woe_data['events']

woe_data['%_events'] = (woe_data['events'] + eps) / (total_events + eps)
woe_data['%_non_events'] = (woe_data['non_events'] + eps) / (total_non_events + eps)

woe_data['WoE'] = np.log(woe_data['%_non_events'] / woe_data['%_events'])

# Маппинг
woe_map = dict(zip(woe_data['city_group'], woe_data['WoE']))
df['city_WoE'] = df['city_group'].map(woe_map)

Частоты городов (топ 10):
city
Mumbai       4392
Delhi        4042
Bengaluru    3279
Hyderabad    2667
Pune         2208
Lucknow      1936
Jaipur       1927
Kochi        1549
Name: count, dtype: int64


In [11]:
# Вывод таблицы WoE
print("WoE по группам городов:")
(woe_data[['city_group', 'count', 'events', 'non_events', 'WoE']])

WoE по группам городов:


,city_group,count,events,non_events,WoE
0,Bengaluru,3279,1476,1803,0.023423
1,Delhi,4042,1796,2246,0.046902
2,Hyderabad,2667,1222,1445,-0.009074
3,Jaipur,1927,886,1041,-0.015495
4,Kochi,1549,714,835,-0.020184
5,Lucknow,1936,841,1095,0.087149
6,Mumbai,4392,2090,2302,-0.080039
7,Pune,2208,1006,1202,0.001292


In [12]:
# Пример закодированных строк
print("\nПример закодированных строк:")
df[['city_group', 'avg_daily_revenue_inr', 'high_revenue', 'city_WoE']].head(10)


Пример закодированных строк:


,city_group,avg_daily_revenue_inr,high_revenue,city_WoE
0,Delhi,332.20,0,0.046902
1,Kochi,3138.90,1,-0.020184
2,Hyderabad,1626.36,1,-0.009074
3,Bengaluru,3592.98,1,0.023423
4,Delhi,2418.07,1,0.046902
5,Pune,1913.73,1,0.001292
6,Mumbai,1475.99,0,-0.080039
7,Lucknow,1004.87,0,0.087149
8,Bengaluru,1026.87,0,0.023423
9,Mumbai,2149.48,1,-0.080039


На основе проведённого анализа для датасета с продавцами еды получены следующие ключевые результаты:

**Бинарная целевая переменная** `high_revenue` создана на основе медианы средней дневной выручки (`1516.59 INR`): ~54.4% объектов - «низкая выручка» (0), ~45.6% - «высокая выручка» (1).

**WoE-кодирование по городам** показало:
  - Наиболее положительный WoE у **Bengaluru** (+0.0234) и **Delhi** (+0.0469) - в этих городах относительно больше точек с *низкой* выручкой.
  - Наиболее отрицательный WoE у **Hyderabad** (−0.0091) и **Mumbai** (−0.0080) - здесь выше доля точек с *высокой* выручкой.
  - Значения WoE близки к нулю, что указывает на слабую, но статистически значимую связь между городом и уровнем выручки.

В примере закодированных строк видно, что один и тот же город (например, Bengaluru) может иметь как высокую, так и низкую выручку, но его WoE-значение остаётся постоянным - это корректное поведение при групповом кодировании.

Вывод: WoE позволяет количественно оценить ассоциацию городов с уровнем выручки и подготовить категориальный признак для использования в моделях машинного обучения.

#### Номер задачи №2 (Задача №23)
#### Для набора данных для одного (произвольного) числового признака проведите обнаружение и удаление выбросов на основе правила трех сигм.

In [15]:
# Общая информация о датасете
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22000 entries, 0 to 21999
Data columns (total 23 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   vendor_id                        22000 non-null  object 
 1   city                             22000 non-null  object 
 2   zone_type                        22000 non-null  object 
 3   vendor_age_years                 22000 non-null  float64
 4   years_in_business                22000 non-null  float64
 5   food_category                    22000 non-null  object 
 6   license_status                   22000 non-null  object 
 7   avg_daily_revenue_inr            22000 non-null  float64
 8   avg_daily_customers              22000 non-null  float64
 9   monthly_stall_rent_inr           22000 non-null  float64
 10  num_helpers                      22000 non-null  int64  
 11  hours_open_per_day               22000 non-null  float64
 12  competition_within

In [16]:
# Выбираем числовой признак
feature = 'avg_daily_customers'

# Вычисляем среднее и стандартное отклонение
mean = df[feature].mean()
std = df[feature].std()

# Определяем границы для выбросов
lower_bound = mean - 3 * std
upper_bound = mean + 3 * std

# Обнаружение выбросов
outliers = df[(df[feature] < lower_bound) | (df[feature] > upper_bound)]
print(f"Количество выбросов: {len(outliers)}")

Количество выбросов: 618


In [17]:
# Удаление выбросов
df_cleaned = df[(df[feature] >= lower_bound) & (df[feature] <= upper_bound)]

# Проверка результатов
print(f"Исходный размер датасета: {len(df)}")
print(f"Размер датасета после удаления выбросов: {len(df_cleaned)}")
print(f"Процент удаленных строк: {100 * (len(df) - len(df_cleaned)) / len(df):.2f}%")

Исходный размер датасета: 22000
Размер датасета после удаления выбросов: 21382
Процент удаленных строк: 2.81%


In [18]:
# Вывод первых 5 строк очищенного датасета
print("\nПервые 5 строк очищенного датасета:")
df_cleaned.head()


Первые 5 строк очищенного датасета:


,vendor_id,city,zone_type,vendor_age_years,years_in_business,food_category,license_status,avg_daily_revenue_inr,avg_daily_customers,monthly_stall_rent_inr,...,monthly_health_inspection_score,had_fine_last_year,avg_monthly_rainfall_mm,season_of_observation,has_online_presence,customer_complaint_rate,vendor_survived,high_revenue,city_group,city_WoE
0,VSF-00001,Delhi,Industrial,34.0,10.98,Chinese,Licensed,332.20,10.0,0.00,...,65.00,0.0,103.08,Winter,0.0,0.12,1,0,Delhi,0.046902
1,VSF-00002,Kochi,University Area,33.0,5.53,Chaat,Licensed,3138.90,180.0,2112.10,...,57.42,1.0,350.00,Monsoon,1.0,0.22,1,1,Kochi,-0.020184
2,VSF-00003,Hyderabad,Industrial,48.0,2.70,Desserts & Sweets,Licensed,1626.36,67.0,5487.75,...,57.42,0.0,66.64,Post-Monsoon,0.0,0.09,0,1,Hyderabad,-0.009074
3,VSF-00004,Bengaluru,Transit Hub,37.0,9.77,Beverages,Licensed,3592.98,171.0,1960.79,...,36.38,1.0,107.30,Post-Monsoon,1.0,0.10,1,1,Bengaluru,0.023423
4,VSF-00005,Delhi,Transit Hub,51.0,21.24,South Indian,Licensed,2418.07,105.0,2720.82,...,71.05,0.0,166.57,Monsoon,0.0,0.08,0,1,Delhi,0.046902


Обнаружение и удаление выбросов в числовом признаке **`avg_daily_customers`** (среднее количество клиентов в день) по правилу трёх сигм дало следующие результаты:

- Обнаружено **618 выбросов** (значения вне интервала `[μ − 3σ, μ + 3σ]`). 
- Исходный размер датасета: **22 000** строк.
- После удаления выбросов осталось: **21 382** строки.
- Доля удалённых строк: **2.81%** - умеренная, что указывает на наличие некоторого числа аномальных значений (например, нулевых или сверхвысоких показателей клиентопотока), но не критичную для целостности выборки.

Таким образом, правило трёх сигм позволило очистить данные от экстремальных наблюдений, повышая надёжность последующего анализа и моделирования без существенной потери информации.